In [10]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['AmazonLaptops'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [11]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Mercado Laboral" # <--- 
datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado correctamente.")

    # --- NUEVO: PROTOCOLO DE INTERVENCIÓN MANUAL (GUÍA 7.2) ---
    driver.get("https://www.amazon.es/s?k=laptops")
    
    print("\n⚠️ ESPERANDO INTERVENCIÓN EN VNC:")
    print("1. Ve a la pestaña de VNC (puerto 6080).")
    print("2. Resuelve el CAPTCHA si aparece.")
    print("3. Cuando veas la lista de laptops en VNC, vuelve aquí.")
    
    input("✅ Presiona ENTER aquí cuando el navegador en VNC esté listo...")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        time.sleep(5) # Espera a que cargue la página

        # Esperar a que los bloques de productos sean visibles
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[data-component-type='s-search-result']"))
        )

        bloques = driver.find_elements(By.CSS_SELECTOR, "div[data-component-type='s-search-result']")

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                # Intentamos capturar el precio (si no tiene, ponemos 0)
                try:
                    precio_text = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text
                except:
                    precio_text = "0"

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio_text,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                continue

        # Intentar avanzar de página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
        except:
            print("Fin de las páginas disponibles.")
            break

    print(f"✅ Extracción terminada: {len(datos_finales)} productos encontrados.")

except Exception as e:
    print(f"❌ Error en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    # Usamos 'mongodb' como host porque así se llama tu servicio en Docker
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpieza de precio para convertir a número
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print("📂 ¡Éxito! Datos guardados en MongoDB.")
    else:
        print("⚠️ No se extrajeron datos, nada que guardar.")

except Exception as e:
    print(f"❌ Error en MongoDB: {e}")

🧹 Limpieza de procesos completada.
🚀 Navegador iniciado correctamente.

⚠️ ESPERANDO INTERVENCIÓN EN VNC:
1. Ve a la pestaña de VNC (puerto 6080).
2. Resuelve el CAPTCHA si aparece.
3. Cuando veas la lista de laptops en VNC, vuelve aquí.


✅ Presiona ENTER aquí cuando el navegador en VNC esté listo... 


--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
✅ Extracción terminada: 180 productos encontrados.
📂 ¡Éxito! Datos guardados en MongoDB.


In [12]:
from pymongo import MongoClient
client = MongoClient("mongodb", 27017)
db = client["TiendaBigData"]
# Muestra los primeros 3 productos guardados
for doc in db["AmazonLaptops"].find().limit(3):
    print(doc)

{'_id': ObjectId('69df9c7d0193ee0a91a619f0'), 'identificador': '18.5" Laptop, 24GB RAM, 1TB SSD, Dual Core Processor m3-8100Y up to 3.4GHz, HDMI, RJ45, Large Screen, Win 11, Telework, Multitasking and Entertainment.', 'valor': 489.0, 'fecha_captura': '2026-04-15 14:10:54', 'grupo': 'Mercado Laboral'}
{'_id': ObjectId('69df9c7d0193ee0a91a619f1'), 'identificador': 'Ryzen 5 7430U 15.6 Inch Laptop PC 16GB RAM 512GB SSD, Laptop Graphics Radeon, FHD IPS, Type-C, USB 3.0, HDMI, Backlit Digital Keyboard Laptop', 'valor': 489.0, 'fecha_captura': '2026-04-15 14:10:54', 'grupo': 'Mercado Laboral'}
{'_id': ObjectId('69df9c7d0193ee0a91a619f2'), 'identificador': 'Laptop 15.6 Inch 16GB RAM 512GB SSD Celeron N5095 Laptop, PC Laptop FHD IPS 1920x1080 Win11 with Backlit Keyboard, Fingerprint Unlock Wi-Fi 5 Laptop', 'valor': 349.0, 'fecha_captura': '2026-04-15 14:10:54', 'grupo': 'Mercado Laboral'}
